# Phase 2d: CLAP sanity check (zero-shot mood classification)

Review comment: CLAP scores below handcrafted features in retrieval (P@10 0.380 vs 0.500) and its linear quadrant probe
reaches only 0.43–0.50, so a reviewer may suspect a setup problem. This notebook (1) records the CLAP checkpoint's
feature-extractor and fusion settings that Phases 2 and 2b used, and (2) measures CLAP's own zero-shot emotion
recognition on MERGE: the stored audio embeddings (projected into CLAP's joint audio–text space) are compared with
CLAP text embeddings of mood prompts.

* **Quadrant zero-shot:** each quadrant is described by several prompts; the prompt embeddings are averaged (prompt
  ensembling) and each track is assigned to the most similar quadrant. Chance is about 0.25.
* **AV zero-shot:** arousal score = similarity to high-arousal prompts minus similarity to low-arousal prompts; valence
  score likewise with positive vs negative prompts. Spearman correlation with MERGE's AV values.

Evaluated for the random-crop CLAP embedding (Phase 2) and the three-fixed-window embedding (Phase 2b), on the 2,075
catalog tracks and on all 3,232 audio tracks. Labels are evaluation targets only. Output: `clap_zero_shot.csv`,
`clap_zero_shot_av.csv`, `clap_zero_shot_confusion.csv`, `clap_settings.json`.

In [1]:
import os, json
os.environ.setdefault('HF_HUB_OFFLINE', '1')          # use the cached checkpoint, as in Phases 2 and 2b
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import ClapModel, ClapProcessor

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
QUADS   = ['Q1', 'Q2', 'Q3', 'Q4']

model = ClapModel.from_pretrained('laion/larger_clap_music').eval()
proc  = ClapProcessor.from_pretrained('laion/larger_clap_music')
fe = proc.feature_extractor
settings = {'checkpoint': 'laion/larger_clap_music', 'sampling_rate': fe.sampling_rate, 'max_length_s': fe.max_length_s,
            'truncation': fe.truncation, 'padding': fe.padding, 'enable_fusion': bool(model.config.audio_config.enable_fusion),
            'projection_dim': model.config.projection_dim}
print(settings)
json.dump(settings, open(os.path.join(OUT_DIR, 'clap_settings.json'), 'w'), indent=1)

merged = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet')).set_index('song_id')
a_old  = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features.parquet')).set_index('song_id')
a_det  = pd.read_parquet(os.path.join(OUT_DIR, 'audio_features_clap_det.parquet')).set_index('song_id')
cat    = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))['song_id'].tolist()
all_ids = a_old.index.tolist()
def l2(X):
    X = np.asarray(X, dtype=np.float64); return X / np.linalg.norm(X, axis=1, keepdims=True)
AUDIO = {'CLAP random crop (Phase 2)': lambda ids: l2(np.vstack(a_old.loc[ids, 'audio_clap'].values)),
         'CLAP three fixed windows (Phase 2b)': lambda ids: l2(np.vstack(a_det.loc[ids, 'audio_clap_det'].values))}

{'checkpoint': 'laion/larger_clap_music', 'sampling_rate': 48000, 'max_length_s': 10, 'truncation': 'rand_trunc', 'padding': 'repeatpad', 'enable_fusion': False, 'projection_dim': 512}


In [2]:
PROMPTS = {
    'Q1': ['happy music', 'upbeat, joyful music', 'cheerful and energetic music', 'exciting, euphoric music'],
    'Q2': ['angry music', 'aggressive, tense music', 'furious and intense music', 'anxious, menacing music'],
    'Q3': ['sad music', 'melancholic, depressing music', 'sorrowful and gloomy music', 'lonely, heartbroken music'],
    'Q4': ['calm music', 'relaxing, peaceful music', 'serene and soothing music', 'tender, content music'],
}
AV_PROMPTS = {
    'arousal_high': ['energetic music', 'intense, exciting music', 'fast, loud music'],
    'arousal_low':  ['calm music', 'quiet, slow music', 'soft, sleepy music'],
    'valence_pos':  ['happy music', 'positive, joyful music', 'pleasant, uplifting music'],
    'valence_neg':  ['sad music', 'negative, dark music', 'unpleasant, distressing music'],
}
def text_emb(texts):
    with torch.no_grad():
        t = proc(text=texts, return_tensors='pt', padding=True)
        return l2(model.get_text_features(**t).numpy())
def ensemble(texts):
    E = text_emb(texts).mean(0); return E / np.linalg.norm(E)
TQ = np.vstack([ensemble(PROMPTS[q]) for q in QUADS])
TAV = {k: ensemble(v) for k, v in AV_PROMPTS.items()}
print('cosine between quadrant prompt centroids:'); print(pd.DataFrame(TQ @ TQ.T, index=QUADS, columns=QUADS).round(3))

cosine between quadrant prompt centroids:
     Q1   Q2   Q3   Q4
Q1  1.0  1.0  1.0  1.0
Q2  1.0  1.0  1.0  1.0
Q3  1.0  1.0  1.0  1.0
Q4  1.0  1.0  1.0  1.0


In [3]:
rows, avrows, conf = [], [], []
for subset, ids in [('catalog (2,075)', cat), ('all audio (3,232)', all_ids)]:
    y = merged.loc[ids, 'quadrant'].values
    Y = merged.loc[ids, ['arousal', 'valence']].values
    for name, get in AUDIO.items():
        A = get(ids)
        S = A @ TQ.T
        pred = np.array(QUADS)[S.argmax(1)]
        acc = accuracy_score(y, pred)
        rng = np.random.default_rng(42)
        bs = [accuracy_score(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(2000))]
        rows.append({'subset': subset, 'embedding': name, 'n': len(ids), 'zero_shot_quadrant_acc': acc,
                     'acc_lo': np.percentile(bs, 2.5), 'acc_hi': np.percentile(bs, 97.5),
                     'macro_F1': f1_score(y, pred, average='macro'),
                     'majority_class_rate': pd.Series(y).value_counts(normalize=True).max(),
                     **{f'recall_{q}': np.mean(pred[y == q] == q) for q in QUADS},
                     **{f'predicted_share_{q}': np.mean(pred == q) for q in QUADS}})
        cm = pd.DataFrame(confusion_matrix(y, pred, labels=QUADS), index=QUADS, columns=QUADS)
        cm.insert(0, 'embedding', name); cm.insert(0, 'subset', subset); conf.append(cm.rename_axis('true').reset_index())
        ar = A @ TAV['arousal_high'] - A @ TAV['arousal_low']
        va = A @ TAV['valence_pos'] - A @ TAV['valence_neg']
        avrows.append({'subset': subset, 'embedding': name,
                       'spearman_arousal': spearmanr(ar, Y[:, 0])[0], 'spearman_valence': spearmanr(va, Y[:, 1])[0],
                       'p_arousal': spearmanr(ar, Y[:, 0])[1], 'p_valence': spearmanr(va, Y[:, 1])[1]})
zs, av = pd.DataFrame(rows), pd.DataFrame(avrows)
zs.to_csv(os.path.join(OUT_DIR, 'clap_zero_shot.csv'), index=False)
av.to_csv(os.path.join(OUT_DIR, 'clap_zero_shot_av.csv'), index=False)
pd.concat(conf).to_csv(os.path.join(OUT_DIR, 'clap_zero_shot_confusion.csv'), index=False)
pd.set_option('display.width', 250)
print(zs.round(3).to_string(index=False)); print(); print(av.round(3).to_string(index=False)); print()
print(pd.concat(conf).to_string(index=False))

           subset                           embedding    n  zero_shot_quadrant_acc  acc_lo  acc_hi  macro_F1  majority_class_rate  recall_Q1  recall_Q2  recall_Q3  recall_Q4  predicted_share_Q1  predicted_share_Q2  predicted_share_Q3  predicted_share_Q4
  catalog (2,075)          CLAP random crop (Phase 2) 2075                   0.217   0.200   0.235     0.146                0.281      0.432        0.0        0.0      0.459               0.450                 0.0                 0.0               0.550
  catalog (2,075) CLAP three fixed windows (Phase 2b) 2075                   0.215   0.197   0.233     0.145                0.281      0.472        0.0        0.0      0.410               0.490                 0.0                 0.0               0.510
all audio (3,232)          CLAP random crop (Phase 2) 3232                   0.224   0.210   0.239     0.150                0.250      0.452        0.0        0.0      0.444               0.475                 0.0                 0.0     

## Reference points
For comparison: CLAP linear quadrant probes reached 0.43–0.50 test accuracy and CLAP retrieval P@10 0.38–0.40 (Table 3);
MuQ probes 0.73–0.77, SBERT 0.72. A correctly set-up CLAP should be clearly above chance zero-shot if its joint space
carries emotion.